# ViTrueVoice — Pipeline Tiền Xử Lý Dữ Liệu & Sinh Spoof Trên Kaggle GPU

Hệ thống phát hiện giọng nói giả tiếng Việt qua kênh thoại nén.
Notebook thực thi trọn vẹn:
1. Chuẩn hóa âm thanh Bonafide 16kHz mono PCM.
2. Tự động sinh dữ liệu Spoof qua F5-TTS và VieNeu-TTS (Adaptive mode) kèm nén viễn thông G.711 / AMR-NB.
3. Lập bảng manifest 17 cột theo `DataColunm_Plan.docx` và phân vùng Speaker-Disjoint (Train, Dev, Test).
4. Đóng gói kết quả về file ZIP để tải về máy local qua `manage_kaggle.py output`.


In [ ]:
# 1. Cài đặt các thư viện cần thiết trên Kaggle GPU
import subprocess
import sys
import os

print('📦 Đang cài đặt thư viện phụ thuộc (F5-TTS, VieNeu-TTS, SpeechBrain, Codecs)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'f5-tts', 'vieneu', 'speechbrain', 'jiwer', 'soundfile', 'librosa', 'pyloudnorm'], check=False)
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libopencore-amrnb-dev'], check=False)

# Kiểm tra bộ giải mã ffmpeg
res = subprocess.run(['ffmpeg', '-encoders'], capture_output=True, text=True)
has_amr = 'libopencore_amrnb' in res.stdout
has_alaw = 'pcm_alaw' in res.stdout
print(f'🔍 Kiểm tra FFmpeg: AMR-NB (libopencore_amrnb)={has_amr}, G.711 A-law (pcm_alaw)={has_alaw}')
if not has_amr:
    print('⚠️ CẢNH BÁO: ffmpeg chưa có libopencore_amrnb. Thử cài đặt gói codecs bổ sung...')
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libavcodec-extra'], check=False)

import torch
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CẢNH BÁO: Không có GPU! Pipeline sẽ chạy rất chậm.')
print('=' * 60)


In [ ]:
# 2. Khởi tạo package vitruevoice trên Kaggle container
import base64
import zlib
import json
import sys
from pathlib import Path

PKG_B64 = '...'

pkg_data = json.loads(zlib.decompress(base64.b64decode(PKG_B64.encode('ascii'))).decode('utf-8'))
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./out')
pkg_root = working_dir / 'vitruevoice'
pkg_root.mkdir(parents=True, exist_ok=True)

for rel_path, content in pkg_data.items():
    file_path = pkg_root / rel_path
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding='utf-8')

if str(working_dir) not in sys.path:
    sys.path.insert(0, str(working_dir))

print(f'✅ Đã khởi tạo thành công package vitruevoice tại: {pkg_root} ({len(pkg_data)} files)')


In [ ]:
# 3. Tiền xử lý Bonafide Audio: Khử trùng văn bản theo thứ tự Test->Dev->Train, Paired Triplets cho Eval
import json
import re
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import subprocess
from pathlib import Path
from vitruevoice.batch_processor import DatasetPartitioner, TelephonyCompressor
from vitruevoice.text_utils import fix_tone_token, normalize_for_eval, has_digits

# ==============================================================
# CẤU HÌNH THỰC THI
# ==============================================================
SMOKE_TEST = False
MAX_VIVOS_SPEAKERS = 65  # Toàn bộ 65 người nói VIVOS
MAX_CV_SPEAKERS = 25     # Tối đa 25 người nói chất lượng cao nhất Common Voice
MAX_PROMPTS_PER_SPK = 15 # 15 câu độc lập / người nói để đạt >= 2.000 mẫu spoof độc lập

data_root = working_dir / 'data'
raw_out_dir = data_root / 'raw'
ref_out_dir = data_root / 'bonafide_ref'
manifests_dir = data_root / 'manifests'
checkpoints_dir = working_dir / 'checkpoints'

raw_out_dir.mkdir(parents=True, exist_ok=True)
ref_out_dir.mkdir(parents=True, exist_ok=True)
manifests_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# --- 1. TÌM KIẾM DATASET VIVOS & COMMON VOICE TRÊN KAGGLE ---
input_roots = [Path('/kaggle/input'), Path('./data')]

# A. Tìm VIVOS
vivos_root = None
for r in input_roots:
    if r.exists():
        cand = list(r.glob('**/vivos')) + list(r.glob('**/prompts.txt'))
        if cand:
            vivos_root = cand[0].parent if cand[0].is_dir() else cand[0].parent.parent
            break
if not vivos_root:
    raise FileNotFoundError('❌ Không tìm thấy dataset VIVOS tại /kaggle/input!')
print(f'📂 Tìm thấy dataset VIVOS tại: {vivos_root}')

# B. Tìm Common Voice 8.0 (Khóa cứng không fallback)
cv_tsv = None
for r in input_roots:
    if r.exists():
        cand = list(r.glob('**/cv-corpus*/vi/validated.tsv')) or list(r.glob('**/validated.tsv'))
        if cand:
            cv_tsv = cand[0]
            break
if not cv_tsv:
    raise FileNotFoundError('❌ Không tìm thấy validated.tsv của Common Voice trong /kaggle/input!')

try:
    cv_dataset_root = next(p for p in [cv_tsv, *cv_tsv.parents] if p.parent == Path('/kaggle/input'))
except StopIteration:
    raise FileNotFoundError('❌ Không xác định được dataset_root hợp lệ của Common Voice dưới /kaggle/input!')

print(f'📂 Common Voice TSV: {cv_tsv}')
print(f'📂 Common Voice Root: {cv_dataset_root}')
print('🔍 Đang lập chỉ mục toàn bộ file .mp3 của Common Voice...')
mp3_index = {f.name: f for f in cv_dataset_root.rglob('*.mp3')}
print(f'📁 Đã lập chỉ mục {len(mp3_index):,} file .mp3.')

# Đo bitrate thực tế của clip Common Voice bằng ffprobe
cv_bitrate = '48k'
sample_mp3 = next(iter(mp3_index.values()), None)
if sample_mp3:
    try:
        probe_res = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'stream=bit_rate', '-of', 'default=noprint_wrappers=1:nokey=1', str(sample_mp3)],
            capture_output=True, text=True
        )
        br_val = int(probe_res.stdout.strip())
        cv_bitrate = f'{br_val // 1000}k'
    except Exception:
        pass
print(f'🎧 Bitrate MP3 thực tế phát hiện được của Common Voice: {cv_bitrate}')

# Hàm tiền xử lý sóng âm: resample 16kHz mono, cắt lặng top_db=25 với 150ms đệm, chuẩn hóa loudness
def standardize_audio(src_path, dst_path, target_sr=16000):
    wav, sr = librosa.load(src_path, sr=target_sr, mono=True)
    if len(wav) > 512:
        _, idx = librosa.effects.trim(wav, top_db=25, frame_length=512, hop_length=160)
        margin = int(target_sr * 0.15)
        start = max(0, idx[0] - margin)
        end = min(len(wav), idx[1] + margin)
        if end > start:
            wav = wav[start:end]
    peak = np.max(np.abs(wav))
    if peak > 1e-6:
        wav = wav * (0.95 / max(peak, 1.0))
    Path(dst_path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(dst_path), wav, target_sr, subtype='PCM_16')
    dur = len(wav) / target_sr
    return round(dur, 4)

# Nạp VIVOS Prompts & lọc câu chứa chữ số
all_transcripts_raw = {}
vivos_prompts = {}
for p_file in vivos_root.rglob('prompts.txt'):
    with open(p_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)
            if len(parts) == 2:
                raw_t = parts[1].strip()
                if not has_digits(raw_t):
                    vivos_prompts[parts[0]] = raw_t
                    all_transcripts_raw[parts[0]] = raw_t
print(f'📖 Đã nạp {len(vivos_prompts)} câu transcript VIVOS sạch (không chữ số).')

# Nạp Common Voice Prompts với bộ lọc nghiêm ngặt (up_votes >= 2, down_votes == 0, không số)
cv_df = pd.read_csv(cv_tsv, sep='\t', low_memory=False)
valid_mask = (
    (cv_df['up_votes'] >= 2) &
    (cv_df['down_votes'] == 0) &
    (cv_df['sentence'].astype(str).str.len() > 10) &
    (~cv_df['sentence'].apply(has_digits))
)
valid_cv_df = cv_df[valid_mask].copy()

cv_prompts = {}
cv_speaker_files = {}
matched_count = 0
for _, row in valid_cv_df.iterrows():
    fname = Path(str(row['path'])).name
    clip_path = mp3_index.get(fname)
    if clip_path and clip_path.exists():
        matched_count += 1
        stem = clip_path.stem
        spk_id = f"CV_{str(row['client_id'])[:10]}"
        raw_sent = str(row['sentence']).strip()
        cv_prompts[stem] = raw_sent
        all_transcripts_raw[stem] = raw_sent
        cv_speaker_files.setdefault(spk_id, []).append(clip_path)

match_ratio = matched_count / max(1, len(valid_cv_df))
print(f'📊 Tỷ lệ khớp file Common Voice: {matched_count}/{len(valid_cv_df)} ({match_ratio*100:.2f}%)')
assert match_ratio >= 0.95, f'❌ LỖI NGHIÊM TRỌNG: Chỉ khớp được {match_ratio*100:.1f}% file audio Common Voice!'
print(f'📖 Đã nạp {len(cv_prompts)} câu transcript và {len(cv_speaker_files)} người nói Common Voice.')

# Thu thập và phân loại người nói
speaker_files_map = {}
for wf in sorted(vivos_root.rglob('*.wav')):
    spk = wf.parent.name
    if wf.stem in all_transcripts_raw:
        speaker_files_map.setdefault(spk, []).append(wf)

for spk, c_files in cv_speaker_files.items():
    if len(c_files) >= 8:
        speaker_files_map[spk] = c_files

vivos_spks = sorted([s for s in speaker_files_map if not s.startswith('CV_')])[:MAX_VIVOS_SPEAKERS]
cv_spks = sorted([s for s in speaker_files_map if s.startswith('CV_')], key=lambda s: len(speaker_files_map[s]), reverse=True)[:MAX_CV_SPEAKERS]

# Phân chia Speaker-Disjoint với tập Test In-domain VIVOS đủ rộng (15-18 speaker)
rng = np.random.RandomState(42)
vivos_shuffled = np.array(vivos_spks)
rng.shuffle(vivos_shuffled)
cv_shuffled = np.array(cv_spks)
rng.shuffle(cv_shuffled)

# VIVOS: 40 Train, 10 Dev, 15 Test In-domain
train_vivos = set(vivos_shuffled[:40])
dev_vivos = set(vivos_shuffled[40:50])
test_vivos = set(vivos_shuffled[50:])

# Common Voice: 15 Train, 5 Dev, 5 Test (Out-of-Domain)
n_cv = len(cv_shuffled)
n_cv_train = max(1, int(n_cv * 0.6))
n_cv_dev = max(1, int(n_cv * 0.2))
train_cv = set(cv_shuffled[:n_cv_train])
dev_cv = set(cv_shuffled[n_cv_train:n_cv_train + n_cv_dev])
test_cv = set(cv_shuffled[n_cv_train + n_cv_dev:])

train_spks = train_vivos | train_cv
dev_spks = dev_vivos | dev_cv
test_spks = test_vivos | test_cv
selected_speakers = sorted(list(train_spks | dev_spks | test_spks))

print(f'👥 Phân chia Speaker: Train={len(train_spks)} (VIVOS:{len(train_vivos)}, CV:{len(train_cv)}), Dev={len(dev_spks)}, Test={len(test_spks)} (In-Domain VIVOS:{len(test_vivos)}, OOD CV:{len(test_cv)})')
assert train_spks.isdisjoint(dev_spks), 'Rò rỉ Train và Dev!'
assert train_spks.isdisjoint(test_spks), 'Rò rỉ Train và Test!'
assert dev_spks.isdisjoint(test_spks), 'Rò rỉ Dev và Test!'

# Thu thập và phân chia Ref, Enroll, Eval cho từng người nói
spk_partitions = {}
for spk in selected_speakers:
    files = speaker_files_map[spk]
    file_durs = {}
    for f in files:
        try:
            file_durs[str(f)] = sf.info(str(f)).duration
        except Exception:
            file_durs[str(f)] = 0.0
    ref_c, enr_c, eval_c = DatasetPartitioner.partition_speaker_files(
        files, n_ref_cands=3, n_enroll=4, file_durations=file_durs
    )
    spk_partitions[spk] = {'ref': ref_c, 'enroll': enr_c, 'eval': eval_c}

# Khử trùng lặp văn bản Text-Disjoint theo thứ tự ưu tiên: Giữ ở Test -> Xóa khỏi Dev -> Xóa khỏi Train
raw_text_by_split = {'train': set(), 'dev': set(), 'test': set()}
for spk, p_dict in spk_partitions.items():
    split_name = 'train' if spk in train_spks else ('dev' if spk in dev_spks else 'test')
    for f in p_dict['eval']:
        t_norm = normalize_for_eval(all_transcripts_raw.get(f.stem, ''))
        if t_norm:
            raw_text_by_split[split_name].add(t_norm)

allowed_test_texts = set(raw_text_by_split['test'])
allowed_dev_texts = set(raw_text_by_split['dev']) - allowed_test_texts
allowed_train_texts = set(raw_text_by_split['train']) - (allowed_test_texts | allowed_dev_texts)

print('📊 Thống kê văn bản sau khử trùng lặp (Text-Disjoint):')
print(f" - Test : {len(allowed_test_texts)} câu giữ nguyên")
print(f" - Dev  : {len(allowed_dev_texts)} câu (loại {len(raw_text_by_split['dev']) - len(allowed_dev_texts)} trùng Test)")
print(f" - Train: {len(allowed_train_texts)} câu (loại {len(raw_text_by_split['train']) - len(allowed_train_texts)} trùng Dev/Test)")

assert allowed_train_texts.isdisjoint(allowed_dev_texts), 'Rò rỉ câu Train và Dev!'
assert allowed_train_texts.isdisjoint(allowed_test_texts), 'Rò rỉ câu Train và Test!'
assert allowed_dev_texts.isdisjoint(allowed_test_texts), 'Rò rỉ câu Dev và Test!'

# Tiến hành chuẩn hóa âm thanh, lọc theo dòng và tạo bộ ba ghép cặp (Paired Triplets) cho Bona fide Eval
bonafide_records = []
for spk in selected_speakers:
    spk_split = 'train' if spk in train_spks else ('dev' if spk in dev_spks else 'test')
    spk_source = 'common_voice' if spk.startswith('CV_') else 'vivos'
    pre_codec_val = f'mp3_{cv_bitrate}' if spk_source == 'common_voice' else 'none'
    p_dict = spk_partitions[spk]

    raw_spk_dir = raw_out_dir / spk
    ref_spk_dir = ref_out_dir / spk
    raw_spk_dir.mkdir(parents=True, exist_ok=True)
    ref_spk_dir.mkdir(parents=True, exist_ok=True)

    # Ref và Enroll: 100% giữ nguyên clean_16khz
    ref_enroll_texts = set()
    for f in p_dict['ref'] + p_dict['enroll']:
        stem = f.stem
        role = 'ref' if f in p_dict['ref'] else 'enroll'
        dst_wav = raw_spk_dir / f'{stem}.wav'
        dur = standardize_audio(f, dst_wav)
        if role == 'ref':
            sf.write(str(ref_spk_dir / f'{stem}.wav'), sf.read(str(dst_wav))[0], 16000)

        raw_t = all_transcripts_raw.get(stem, '')
        ref_enroll_texts.add(normalize_for_eval(raw_t))
        bonafide_records.append({
            'utt_id': stem,
            'audio_path': f'data/raw/{spk}/{stem}.wav',
            'speaker_id': spk,
            'source': spk_source,
            'label': 'bonafide',
            'role': role,
            'tts_generator': 'none',
            'codec': 'clean_16khz',
            'pre_codec': pre_codec_val,
            'split': spk_split,
            'duration_sec': dur,
            'transcript': raw_t,
            'raw_wer': None, 'raw_cer': None, 'ecapa_sim': None, 'wavlm_sim': None,
            'derived_tier': 'BonaFide', 'seed': 100,
        })

    # Eval Bona fide: lọc văn bản trùng và tạo bộ ba ghép cặp (clean, g711, amr)
    allowed_texts_set = allowed_train_texts if spk_split == 'train' else (allowed_dev_texts if spk_split == 'dev' else allowed_test_texts)
    for f in p_dict['eval']:
        stem = f.stem
        raw_t = all_transcripts_raw.get(stem, '')
        norm_t = normalize_for_eval(raw_t)
        # Loại bỏ nếu văn bản bị trùng hoặc trùng với ref/enroll của chính speaker đó
        if (norm_t not in allowed_texts_set) or (norm_t in ref_enroll_texts):
            continue

        # 1. Bản clean_16khz
        dst_clean = raw_spk_dir / f'{stem}_clean.wav'
        dur_clean = standardize_audio(f, dst_clean)

        # 2. Bản G.711 A-law
        dst_g711 = raw_spk_dir / f'{stem}_g711.wav'
        ok_g711 = TelephonyCompressor.compress_g711_alaw(str(dst_clean), str(dst_g711))

        # 3. Bản AMR-NB
        dst_amr = raw_spk_dir / f'{stem}_amrnb.wav'
        ok_amr = TelephonyCompressor.compress_amr_nb(str(dst_clean), str(dst_amr))

        if ok_g711 and ok_amr:
            bonafide_records.append({
                'utt_id': f'{stem}_clean',
                'audio_path': f'data/raw/{spk}/{stem}_clean.wav',
                'speaker_id': spk,
                'source': spk_source,
                'label': 'bonafide',
                'role': 'eval',
                'tts_generator': 'none',
                'codec': 'clean_16khz',
                'pre_codec': pre_codec_val,
                'split': spk_split,
                'duration_sec': dur_clean,
                'transcript': raw_t,
                'raw_wer': None, 'raw_cer': None, 'ecapa_sim': None, 'wavlm_sim': None,
                'derived_tier': 'BonaFide', 'seed': 100,
            })
            bonafide_records.append({
                'utt_id': f'{stem}_g711',
                'audio_path': f'data/raw/{spk}/{stem}_g711.wav',
                'speaker_id': spk,
                'source': spk_source,
                'label': 'bonafide',
                'role': 'eval',
                'tts_generator': 'none',
                'codec': 'g711_alaw',
                'pre_codec': pre_codec_val,
                'split': spk_split,
                'duration_sec': dur_clean,
                'transcript': raw_t,
                'raw_wer': None, 'raw_cer': None, 'ecapa_sim': None, 'wavlm_sim': None,
                'derived_tier': 'BonaFide', 'seed': 100,
            })
            bonafide_records.append({
                'utt_id': f'{stem}_amrnb',
                'audio_path': f'data/raw/{spk}/{stem}_amrnb.wav',
                'speaker_id': spk,
                'source': spk_source,
                'label': 'bonafide',
                'role': 'eval',
                'tts_generator': 'none',
                'codec': 'amr_nb',
                'pre_codec': pre_codec_val,
                'split': spk_split,
                'duration_sec': dur_clean,
                'transcript': raw_t,
                'raw_wer': None, 'raw_cer': None, 'ecapa_sim': None, 'wavlm_sim': None,
                'derived_tier': 'BonaFide', 'seed': 100,
            })

bonafide_df = pd.DataFrame(bonafide_records)
bonafide_csv = manifests_dir / 'bonafide_manifest.csv'
bonafide_df.to_csv(bonafide_csv, index=False, encoding='utf-8')
print(f'✅ Đã lưu {len(bonafide_df)} mẫu Bona fide chuẩn hóa tại: {bonafide_csv}')


In [ ]:
# 4. Sinh giọng Spoof qua Dual Pipeline (kèm MP3 round-trip cho Common Voice OOD & Checkpoints)
import dataclasses
from vitruevoice.batch_processor import BatchDatasetProcessor
from vitruevoice.pipeline import ViTrueVoiceDualPipeline, PipelineMode
from vitruevoice.evaluator import MultiGateEvaluator

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if 'processor' not in globals() or processor is None:
    print(f'🚀 Đang khởi tạo Dual Pipeline và Evaluator trên thiết bị: {device}...')
    evaluator = MultiGateEvaluator(device=device)
    pipeline = ViTrueVoiceDualPipeline(device=device, evaluator=evaluator)
    processor = BatchDatasetProcessor(pipeline=pipeline, evaluator=evaluator, device=device)
else:
    print(f'♻️ Tái sử dụng Pipeline & Evaluator đang có sẵn trên {device}.')

spoof_f5_dir = data_root / 'spoof_f5tts'
spoof_vn_dir = data_root / 'spoof_vieneu'
spoof_f5_dir.mkdir(parents=True, exist_ok=True)
spoof_vn_dir.mkdir(parents=True, exist_ok=True)

# Bản đồ transcript tham chiếu
ref_transcript_map = {**all_transcripts_raw}
all_spoof_records = []

def _rget(obj, names, default=None):
    """Lấy giá trị từ record theo danh sách tên field có thể có."""
    for n in names:
        if hasattr(obj, n):
            v = getattr(obj, n)
            if v is not None:
                return v
    if isinstance(obj, dict):
        for n in names:
            if obj.get(n) is not None:
                return obj[n]
    return default

for spk in selected_speakers:
    ckpt_file = checkpoints_dir / f'{spk}_done.json'
    spk_raw_dir = raw_out_dir / spk
    wav_files = sorted([f.as_posix() for f in spk_raw_dir.glob('*.wav')])
    if not wav_files:
        continue

    # Checkpoint & Resume
    if ckpt_file.exists():
        print(f'⏩ Speaker {spk} đã có checkpoint hoàn tất. Đang đọc lại kết quả...')
        saved_recs = json.loads(ckpt_file.read_text(encoding='utf-8'))
        all_spoof_records.extend(saved_recs)
        continue

    spk_split = 'train' if spk in train_spks else ('dev' if spk in dev_spks else 'test')
    spk_source = 'common_voice' if spk.startswith('CV_') else 'vivos'
    pre_codec_val = f'mp3_{cv_bitrate}' if spk_source == 'common_voice' else 'none'

    # Lựa chọn câu prompt: giữ nguyên văn bản gốc, lọc không số, tuân thủ text-disjoint
    allowed_texts_set = allowed_train_texts if spk_split == 'train' else (allowed_dev_texts if spk_split == 'dev' else allowed_test_texts)
    ref_stems = {f.stem for f in (ref_out_dir / spk).glob('*.wav')}
    spk_bona = bonafide_df[(bonafide_df['speaker_id'] == spk) & (bonafide_df['role'] == 'eval')]
    
    candidate_prompts = []
    for _, r_row in spk_bona.iterrows():
        raw_p = str(r_row['transcript']).strip()
        norm_p = normalize_for_eval(raw_p)
        if (norm_p in allowed_texts_set) and (not has_digits(raw_p)) and (len(raw_p) > 10):
            if raw_p not in candidate_prompts:
                candidate_prompts.append(raw_p)
        if len(candidate_prompts) >= MAX_PROMPTS_PER_SPK:
            break

    if not candidate_prompts:
        candidate_prompts = [
            'xin chào tôi là người nhà cần liên hệ gấp về việc viện phí.',
            'alo bạn ơi chuyển khoản giúp tôi mười triệu vào tài khoản nhé.',
            'chào bạn tôi gọi từ ngân hàng xác nhận giao dịch bảo hiểm.',
            'hôm nay trời đẹp và chúng ta cùng đi dạo trên bờ sông nhé.',
            'anh gọi lại ngay cho em khi nhận được tin nhắn này nhé.'
        ][:MAX_PROMPTS_PER_SPK]

    ref_files = sorted([f.as_posix() for f in (ref_out_dir / spk).glob('*.wav')])
    print(f'\n🎙️ Đang xử lý {spk} ({spk_source}, {len(candidate_prompts)} câu độc lập, 1:1:1 Paired Triplets)...')
    records = processor.process_speaker(
        speaker_id=spk,
        speaker_wav_files=wav_files,
        ref_cands=ref_files,
        prompts=candidate_prompts,
        out_synth_dir=data_root,
        mode=PipelineMode.DUAL_OUTPUT,
        apply_telephony=True,
        only_pass=True,
        transcript_map=ref_transcript_map,
        max_retries=2,
        out_f5_dir=spoof_f5_dir / spk,
        out_vn_dir=spoof_vn_dir / spk
    )

    # Lưu checkpoint cho speaker
    serialized = []
    for i, r in enumerate(records):
        if i == 0:
            fields = dataclasses.asdict(r) if dataclasses.is_dataclass(r) else vars(r)
            print(f'🧩 Các field của ManifestRecord: {list(fields.keys())}')

        w_path = Path(r.audio_path)
        if not w_path.is_absolute():
            w_path = working_dir / w_path

        # MP3 round-trip cho bản clean của Common Voice (OOD)
        if spk_source == 'common_voice' and r.codec == 'clean_16khz' and w_path.exists():
            tmp_mp3 = w_path.with_suffix('.mp3')
            subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', str(w_path),
                            '-c:a', 'libmp3lame', '-b:a', cv_bitrate, str(tmp_mp3)], check=False)
            if tmp_mp3.exists():
                subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', str(tmp_mp3),
                                '-ar', '16000', '-ac', '1', str(w_path)], check=False)
                tmp_mp3.unlink()

        # Thời lượng thực tế thay vì gán cứng 4.0
        try:
            dur = round(sf.info(str(w_path)).duration, 4)
        except Exception:
            dur = 0.0

        # Đường dẫn tương đối, thống nhất với manifest bonafide
        try:
            rel_path = w_path.relative_to(working_dir).as_posix()
        except ValueError:
            rel_path = str(r.audio_path)

        transcript_val = _rget(r, ['transcript', 'text', 'prompt_text', 'prompt', 'gen_text', 'target_text', 'sentence'], '')
        if not transcript_val and '_prompt_' in w_path.stem:
            try:
                p_idx = int(w_path.stem.split('_prompt_')[1].split('_')[0])
                if 0 <= p_idx < len(candidate_prompts):
                    transcript_val = candidate_prompts[p_idx]
            except Exception:
                pass

        rec_dict = {
            'utt_id': w_path.stem,
            'audio_path': rel_path,
            'speaker_id': r.speaker_id,
            'source': spk_source,
            'label': 'spoof',
            'role': 'eval',
            'tts_generator': _rget(r, ['tts_generator', 'engine_name', 'engine', 'generator'], 'F5-TTS'),
            'codec': r.codec,
            'pre_codec': pre_codec_val,
            'split': spk_split,
            'duration_sec': dur,
            'transcript': transcript_val,
            'raw_wer': _rget(r, ['raw_wer']),
            'raw_cer': _rget(r, ['raw_cer']),
            'ecapa_sim': _rget(r, ['ecapa_sim']),
            'wavlm_sim': _rget(r, ['wavlm_sim']),
            'derived_tier': r.derive_attack_tier(),
            'seed': _rget(r, ['seed'], 100),
        }
        serialized.append(rec_dict)

    ckpt_file.write_text(json.dumps(serialized, indent=2, ensure_ascii=False), encoding='utf-8')
    all_spoof_records.extend(serialized)
    print(f'✔️ Hoàn tất {spk}: sinh {len(serialized)} mẫu spoof ({spk_source})')

spoof_df = pd.DataFrame(all_spoof_records)
if not spoof_df.empty:
    empty_t_ratio = spoof_df['transcript'].eq('').mean()
    print(f'🔍 Tỷ lệ transcript rỗng của spoof: {empty_t_ratio:.2%}')
    assert empty_t_ratio < 0.05, f'Lỗi: Quá nhiều transcript rỗng ({empty_t_ratio:.2%})'
spoof_csv = manifests_dir / 'spoof_manifest.csv'
spoof_df.to_csv(spoof_csv, index=False, encoding='utf-8')
print(f'\n✅ Đã hoàn tất sinh spoof! Lưu manifest tại: {spoof_csv} ({len(spoof_df)} mẫu)')


In [ ]:
# 5. Kiểm tra Cấu trúc Ghép cặp Paired Triplets, Trọng số Loss Động & Xuất Manifest 18 Cột
import torch

MANIFEST_COLUMNS = [
    'utt_id', 'audio_path', 'speaker_id', 'source', 'label', 'role',
    'tts_generator', 'codec', 'pre_codec', 'split', 'duration_sec', 'transcript',
    'raw_wer', 'raw_cer', 'ecapa_sim', 'wavlm_sim', 'derived_tier', 'seed'
]

bonafide_std_df = bonafide_df[MANIFEST_COLUMNS].copy() if not bonafide_df.empty else pd.DataFrame(columns=MANIFEST_COLUMNS)
spoof_std_df = pd.DataFrame(all_spoof_records)[MANIFEST_COLUMNS].copy() if all_spoof_records else pd.DataFrame(columns=MANIFEST_COLUMNS)

dfs_to_concat = [df for df in [bonafide_std_df, spoof_std_df] if not df.empty]
full_df = pd.concat(dfs_to_concat, ignore_index=True) if dfs_to_concat else pd.DataFrame(columns=MANIFEST_COLUMNS)

# Kiểm tra cấu trúc bộ ba ghép cặp (Paired Triplets Integer Equality) trên tập Eval
eval_df = full_df[full_df['role'] == 'eval'].copy()
count_clean = (eval_df['codec'] == 'clean_16khz').sum()
count_g711 = (eval_df['codec'] == 'g711_alaw').sum()
count_amr = (eval_df['codec'] == 'amr_nb').sum()
print(f'📊 Kiểm tra số lượng Codec Eval: clean={count_clean}, g711={count_g711}, amr={count_amr}')
assert count_clean == count_g711 == count_amr, f'❌ Lỗi cấu trúc: Tỷ lệ codec eval không bằng nhau ({count_clean}, {count_g711}, {count_amr})!'

full_manifest_csv = manifests_dir / 'full_manifest.csv'
full_df.to_csv(full_manifest_csv, index=False, encoding='utf-8')

train_df = full_df[full_df['split'] == 'train'].copy().reset_index(drop=True)
dev_df = full_df[full_df['split'] == 'dev'].copy().reset_index(drop=True)
test_df = full_df[full_df['split'] == 'test'].copy().reset_index(drop=True)

test_in_df = test_df[test_df['source'] != 'common_voice'].copy().reset_index(drop=True)
test_ood_df = test_df[test_df['source'] == 'common_voice'].copy().reset_index(drop=True)

train_df.to_csv(manifests_dir / 'train.csv', index=False, encoding='utf-8')
dev_df.to_csv(manifests_dir / 'dev.csv', index=False, encoding='utf-8')
test_in_df.to_csv(manifests_dir / 'test_indomain.csv', index=False, encoding='utf-8')
if not test_ood_df.empty:
    test_ood_df.to_csv(manifests_dir / 'test_ood.csv', index=False, encoding='utf-8')

# Tính trọng số Loss động theo nghịch tần suất tập Train
labels_num = train_df['label'].astype(str).str.lower().map({'spoof': 0, 'bonafide': 1}).values
assert not np.isnan(labels_num).any(), 'Có nhãn không hợp lệ trong train_df!'
counts = np.bincount(labels_num, minlength=2)
assert counts.min() > 0, 'Tập train thiếu một trong hai lớp!'
total_train = len(labels_num)
raw_w = total_train / (2.0 * counts)
norm_w = raw_w / np.sum(raw_w)
dynamic_weights = torch.tensor(norm_w, dtype=torch.float32)

# Lưu trọng số Loss vào file JSON
loss_weights_dict = {
    'spoof': round(float(norm_w[0]), 4),
    'bonafide': round(float(norm_w[1]), 4),
    'counts': {'spoof': int(counts[0]), 'bonafide': int(counts[1])},
    'total_train': int(total_train)
}
(manifests_dir / 'loss_weights.json').write_text(json.dumps(loss_weights_dict, indent=2), encoding='utf-8')

print('=' * 60)
print('📊 BÁO CÁO TỔNG KẾT TẬP DỮ LIỆU ĐÃ CHUẨN HÓA (18 CỘT):')
print(f" - Train ({train_df['speaker_id'].nunique()} spks): {len(train_df)} mẫu (Bona: {(train_df['label']=='bonafide').sum()}, Spoof: {(train_df['label']=='spoof').sum()})")
print(f" - Dev   ({dev_df['speaker_id'].nunique()} spks): {len(dev_df)} mẫu (Bona: {(dev_df['label']=='bonafide').sum()}, Spoof: {(dev_df['label']=='spoof').sum()})")
print(f" - Test  ({test_df['speaker_id'].nunique()} spks): {len(test_df)} mẫu (In-Domain: {len(test_in_df)}, OOD: {len(test_ood_df)})")
print(f" - Tổng cộng: {len(full_df)} mẫu")
print(f'⚖️ Trọng số Loss động tập Train [w_spoof, w_bona]: [{norm_w[0]:.4f}, {norm_w[1]:.4f}]')
print('\nPhân bổ tầng tấn công:')
print(full_df['derived_tier'].value_counts())
print('\nPhân bổ mã hóa (Codec):')
print(full_df['codec'].value_counts())
print('\nPhân bổ nguồn dữ liệu (Source):')
print(full_df['source'].value_counts())
print('\nPhân bổ Pre-codec (MP3 OOD vs None):')
print(full_df['pre_codec'].value_counts())
print('=' * 60)


In [ ]:
# 6. Đóng gói thư mục data thành file ZIP cho Kaggle Output
import zipfile
import shutil

for m in list(manifests_dir.glob('*.csv')) + list(manifests_dir.glob('*.json')):
    shutil.copy(m, working_dir / m.name)

zip_out = working_dir / 'vitruevoice_data_processed.zip'
print(f'📦 Đang nén thư mục data vào {zip_out.name}...')
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in data_root.rglob('*'):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(working_dir))

print(f'✅ Đã tạo gói dữ liệu thành công: {zip_out} ({zip_out.stat().st_size / 1e6:.2f} MB)')
print('Tải kết quả về máy local: python manage_kaggle.py output')
